# 🫀 실험 22-A — **축별 전이 강건성을 같은 자로 잰다**

**MedKOS / `notebooks/exp22a_axis_transfer.ipynb`** · 퀘스트 `ailab-2026-0015`

> 먼저 읽을 것: **`pipelines/SCORING_RULES.md`** (R1~R9)

## 이 실험이 하는 일 — "새로 재기" 가 아니라 "같은 자로 모으기"

축별 전이 결과가 이미 **두 군데 흩어져 있다**:

| 축 | 어디 | 지표 | 문제 |
|---|---|---|---|
| 형태(MI) | 실험20 / 20c / 20d | **AUROC** 낙폭 | — |
| 비트(S·V) | `mit-bih/PAPER.md §6.5` | **PR-AUC** + lift | 코호트 간 비교 불가 |

`PR-AUC` 는 기저율에 붙어 있고, `lift` 는 **천장이 `1/기저율`** 이라 코호트마다 다르다
(MIT-BIH DS2 26.8× vs INCART 89.3× — 3배 차이). **R4 위반**이다.

→ **AUROC 로 다시 뽑아** 축 간 낙폭을 같은 자로 잰다.

## 그리고 §6.5 에는 짝이 없다

§6.5 는 **cross(INCART) 만** 있고 **within 기준선이 같은 모델에서 나오지 않았다**.
낙폭 = `within − cross` 이므로 **같은 학습으로 둘 다** 예측해야 한다.

> 그래서 이 실험은 `colab_crossdb.py` 의 학습 분할을 **MIT-BIH 전체 → DS1** 로 바꾼다.
> 그러면 `DS2`(within)와 `INCART`(cross)를 **같은 모델**로 예측할 수 있다.
> 이건 §6.5 의 **한계 L3**(PID 를 레코드로 셈)도 부분적으로 고친다.

## 사전등록

| 관문 | 내용 | 지지 조건 |
|---|---|---|
| **G0** | 자산·분할 점검 — `pid` 배열 존재 · 환자 vs 레코드 수 · DS1/DS2 누수 0 | 하나라도 실패하면 **중단** |
| **G1** | Drive pkl 10개에서 V/S Δ 재계산 → 미검증 인용을 실측으로 대체 | 재현되면 인용 해금 |
| **P-A★★** | **`V`(형태 정의 비트)의 낙폭 < `S`(타이밍 정의 비트)의 낙폭** | 차 > 0.05 (AUROC) |
| **P-B★★** | 리듬 축을 넣으면 `S` 의 **낙폭이 줄어드나** | `S` 낙폭(v2) < `S` 낙폭(v1) |
| **P-C** | BN 적응이 낙폭을 복구하나 | 복구되면 분포 shift · 안 되면 **판별 축 부재** |
| **P-D** | 축 비교표 — 형태(MI) 낙폭 vs 비트 V·S 낙폭 | 서술 보고(문턱 없음) |

### P-A 가 왜 핵심인가

`V`(심실조기박동)는 **넓은 QRS** 라는 형태로 정의된다 — 기기·인구가 바뀌어도 형태는 형태다.
`S`(심방조기박동)는 **"그 환자 평소보다 이르다"** 는 개인 기준 상대량이다 — 기저 심박수가
다른 인구로 옮기면 기준선이 통째로 이동한다.

**같은 실행 · 같은 모델 · 같은 테스트셋** 안에서 두 클래스를 비교하므로
`V` 가 **양성 대조군** 역할을 한다 — "실험이 망가진 게 아니라 `S` 만 다르게 행동한다".

## 하지 않는 것

- 새 백본 · 하이퍼파라미터 튜닝 · 임계값 조정
- `mit-bih/PAPER.md §6.5` 의 기존 수치 **수정** (병기만 한다)
- 비트 단위와 레코드 단위 **성능의 직접 비교** — 낙폭만 비교한다


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/SCORING_RULES.md R2·R3)
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    """사전등록 관문의 유일한 계약: 지지 / 기각 / **미결**."""
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return True
        if hi < thr: return False
    else:
        if hi < thr: return True
        if lo > thr: return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def t_ci(v, conf=.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2: return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

def t_ci_logit(v, conf=.95):
    """R2 — 유계 지표(0~1)는 logit 에서. 실험20b 가 특이도 CI 를 음수로 냈다."""
    v = np.clip(np.asarray([x for x in v if np.isfinite(x)], float), 1e-6, 1 - 1e-6)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    m, lo, hi = t_ci(np.log(v / (1 - v)), conf)
    f = lambda x: float(1 / (1 + np.exp(-x)))
    return f(m), f(lo), f(hi)

def boot_indices(n, B, seed):
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

class AssetError(RuntimeError):
    """자산·분할이 가정과 다르면 **추측하지 않고 멈춘다**."""

print("사전점검 적재: decide · t_ci · t_ci_logit · boot_indices · AssetError")


In [ ]:
# CELL 1 — 설정
# ★ WST(colab_step12_wst.py) 가 kymatio 를 쓴다. **여기서 미리** 깔아야 한다.
#   colab_step12_wst.py 안에도 자동설치가 있지만 설치 후 importlib.invalidate_caches()
#   를 안 불러서, 같은 세션에서 한 번 실패한 import 는 캐시 때문에 계속 실패한다
#   (같은 프로젝트의 colab_bootstrap.py::_ensure 는 그걸 고쳐놨다 — step12 만 빠졌다).
!pip -q install kymatio

import os, sys, json, time, pickle, subprocess, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")          # colab_crossdb.py 의 _BASE
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SMOKE = False        # ★ True 로 두면 시드 1개만 — 학습 루프가 도는지 먼저 확인한다
SEEDS = [2000] if SMOKE else [2000, 2001, 2002, 2003, 2004]   # crossdb 기본 시드대
KWST, BOOT, SEED0 = 40, 2000, 20260801
GAP_THR = 0.05                               # P-A 문턱 (AUROC 낙폭 차)

# ── 형태 축(MI) 낙폭 — 실험20/20d 실측. **비교 상대**로만 쓴다
MI_DROPS = {"IMI": 0.1246, "ILMI": 0.0877, "ASMI": 0.0908}
MI_NOTE = ("실험20 실측(v2-narrow 채점 3부위). AMI 는 라벨 오류였으므로 제외 "
           "— 실험20c·20d 참조. 내부는 5겹 OOF·외부는 전량 학습이라 **과소추정**")

# ── Drive 자산 (mit-bih/ASSETS_DRIVE.md 에 등록된 것)
PKL_DIR_HINT = "1ZSiLz1xoT-8aHPjE3yc4TBHxOYor2vD2"   # 확률 pkl 10개
PKL_NAMES = [f"mit_only_{1000+i}.pkl" for i in range(5)] + \
            [f"mit_svdb_{1000+i}.pkl" for i in range(5)]

CONFIG = dict(exp="exp22a_axis_transfer", quest="ailab-2026-0015",
              parent_exp=["exp20_ptbdb", "exp20d_combined", "mit-bih/PAPER.md §6.5"],
              purpose=("축별 전이 강건성을 **같은 자(AUROC)** 로 모은다. §6.5 는 PR-AUC 라 "
                       "코호트 간 비교가 안 되고, within 기준선이 같은 모델에서 안 나왔다"),
              dataset="MIT-BIH Arrhythmia (DS1 학습 / DS2 within) + INCART (cross)",
              change_one_thing=("colab_crossdb.py 의 학습 분할을 MIT-BIH 전체 → DS1 로 바꾼다. "
                                "백본·특징·하이퍼파라미터는 그대로"),
              seeds=SEEDS, kwst=KWST, gap_thr=GAP_THR,
              mi_drops=MI_DROPS, mi_note=MI_NOTE,
              metric_rule=("R4 — 코호트를 가로지르는 비교는 **AUROC** 로만 한다. "
                           "PR-AUC 는 기저율에 붙고 lift 는 천장이 1/기저율 이라 "
                           "MIT-BIH DS2(26.8x)와 INCART(89.3x)가 3배 다르다"),
              predictions={
                  "G0": "pid 배열 존재 · 환자 vs 레코드 수 보고 · DS1/DS2 누수 0",
                  "G1": "Drive pkl 10개에서 V/S Δ 재계산 → 미검증 인용 해금",
                  "P-A": f"V 낙폭 < S 낙폭, 차 > {GAP_THR} (AUROC)",
                  "P-B": "리듬 축을 넣으면 S 의 낙폭이 줄어든다",
                  "P-C": "BN 적응이 낙폭을 복구하나 (복구 실패 = 판별 축 부재)",
                  "P-D": "축 비교표 — 형태(MI) vs 비트 V·S. 서술 보고"},
              caveat=("비트 단위와 레코드 단위의 **성능**은 비교하지 않는다 — 낙폭만 비교한다. "
                      "§6.5 의 한계 L1·L2·L4·L5 는 그대로 남는다(L3 만 부분 교정)"))
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp22a_axis", CONFIG, project=PROJECT)
run.log(f"mit-bih 자산 루트: {MITBIH}")
run.log(f"형태 축 비교 상대(실험20): {MI_DROPS}")


In [ ]:
# CELL 2 — 【G0】 자산·분할 점검. **가정이 틀리면 여기서 멈춘다**
#   실험20 의 교훈: 무거운 계산 전에 값싼 관문으로 죽인다.
need = {"mamba_data.npz": "MIT-BIH 비트·라벨·pid",
        "incart_data.npz": "INCART 비트·라벨·pid",
        "colab_crossdb.py": "교차DB 하니스",
        "colab_step12_wst.py": "WST 추출기"}
miss = [f for f in need if not os.path.exists(os.path.join(MITBIH, f))]
if miss:
    raise AssetError(
        f"필요한 자산이 없다: {miss}\n"
        f"  찾은 곳: {MITBIH}\n"
        f"  → mit-bih/ASSETS_DRIVE.md 를 보고 위치를 확인할 것. **추측해서 만들지 않는다.**")
run.log("자산 확인 ✅ " + " · ".join(need))

# ── 패키지 게이트: 무거운 계산 전에 import 가 되는지 **실제로** 해본다
#   실험22-A 첫 실행에서 CELL 4 가 kymatio 없이 여기까지 와서 터졌다.
for _pkg, _why in (("kymatio", "WST 산란변환(colab_step12_wst.py)"),
                   ("torch", "백본"), ("sklearn", "스케일러·특징선택")):
    try:
        importlib.import_module(_pkg)
    except ModuleNotFoundError:
        raise AssetError(
            f"패키지 '{_pkg}' 를 import 할 수 없다 ({_why}).\n"
            f"  → CELL 1 의 `!pip -q install {_pkg}` 가 실패했거나 런타임 재시작이 필요하다.\n"
            "     학습으로 넘어가지 않는다.")
run.log("패키지 확인 ✅ kymatio · torch · sklearn")

_DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
_DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
assert not (set(_DS1) & set(_DS2)), "DS1/DS2 가 겹친다"

run.log("\n【G0】 npz 키 전수 — 가정한 키가 없으면 중단")
KEYS = {}
for f in ("mamba_data.npz", "incart_data.npz"):
    d = np.load(os.path.join(MITBIH, f))
    KEYS[f] = list(d.files)
    run.log(f"  {f:<20} {KEYS[f]}")
for f, want in (("mamba_data.npz", ("beat", "y", "pid", "feats")),
                ("incart_data.npz", ("beat", "y", "pid", "pre_rr", "post_rr"))):
    lack = [k for k in want if k not in KEYS[f]]
    if lack:
        raise AssetError(f"{f} 에 {lack} 이 없다 · 실제 키 {KEYS[f]}\n"
                         "  → 레코드 ID(pid) 없이는 환자 단위 분할을 할 수 없다. 중단한다.")
run.log("  ✅ 가정한 키가 전부 있다")

dm = np.load(os.path.join(MITBIH, "mamba_data.npz"))
di = np.load(os.path.join(MITBIH, "incart_data.npz"))
mpid, my = dm["pid"], dm["y"]; ipid, iy = di["pid"], di["y"]
TR = np.isin(mpid, _DS1); TE = np.isin(mpid, _DS2)
run.log(f"\n  MIT-BIH {len(my):,}비트 · 레코드 {len(np.unique(mpid))}개")
run.log(f"    DS1(학습) {int(TR.sum()):,}비트 / 레코드 {len(np.unique(mpid[TR]))}개")
run.log(f"    DS2(within) {int(TE.sum()):,}비트 / 레코드 {len(np.unique(mpid[TE]))}개")
run.log(f"    미분류 {int((~TR & ~TE).sum()):,}비트 (DS1·DS2 어디에도 없는 레코드)")
if TR.sum() == 0 or TE.sum() == 0:
    raise AssetError("DS1 또는 DS2 가 비었다 — pid 가 레코드 번호가 아닐 수 있다")
leak = set(np.unique(mpid[TR])) & set(np.unique(mpid[TE]))
if leak:
    raise AssetError(f"DS1/DS2 레코드 누수 {leak}")
run.log("  ✅ DS1/DS2 레코드 누수 0")

run.log(f"\n  INCART {len(iy):,}비트 · **pid 고유값 {len(np.unique(ipid))}개**")
run.log("    ⚠️ §6.5 한계 L3 — INCART 실제 환자는 32명인데 pid 를 레코드(75)로 셌다.")
run.log("       이 실험은 INCART 를 **테스트로만** 쓰고 내부 분할을 안 하므로 L3 의")
run.log("       영향은 '환자 단위 CI 를 못 낸다' 로 한정된다. 낙폭 점추정은 유효하다.")
for tag, y in (("MIT-BIH DS1", my[TR]), ("MIT-BIH DS2", my[TE]), ("INCART", iy)):
    n = len(y)
    run.log(f"    {tag:<14} N/S/V = {int((y==0).sum()):>6,}/{int((y==1).sum()):>5,}"
            f"/{int((y==2).sum()):>5,}  · S 기저율 {(y==1).mean():.4f} · "
            f"V 기저율 {(y==2).mean():.4f}")
run.log("\n  ※ 기저율이 코호트마다 다르다 → **PR-AUC·lift 로 비교하지 않는다**(R4)")
_s1, _s2 = float((my[TR] == 1).mean()), float((my[TE] == 1).mean())
run.log(f"  ⚠️ **DS1 S {_s1:.4f} vs DS2 S {_s2:.4f} — {_s2/_s1:.1f}배 차이.** MIT-BIH 고유 성질이다.")
run.log("     즉 'within' 기준선조차 순수한 동일분포 검정이 아니다(유병률 이동이 이미 있다).")
run.log("     AUROC 는 유병률 무관이라 낙폭 계산은 유효하지만, **within 을 '이상적 상한'**")
run.log("     으로 읽으면 안 된다 — 낙폭은 그만큼 **과소추정**된다. 한계로 기록한다.")
CONFIG["within_prevalence_shift"] = {"DS1_S": _s1, "DS2_S": _s2, "ratio": _s2 / max(_s1, 1e-9)}
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【G1】 Drive pkl 회수 (2026-08-02 재작성)
#
#  1차 실행: "proba 길이 47801 != DS2 49295 → 건너뜀". 그 **판정은 옳다** — pkl 에
#  정답 라벨 y 가 없어서 우리 DS2 배열에서 라벨을 빌려오려 했는데, 길이가 안 맞는데
#  잘라 쓰면 조용히 **다른 실험**이 된다. 하지만 거기서 멈추면 둘을 놓친다:
#    ① 1,494 비트가 왜 다른지 — 07-17 실행의 정체를 규정하는 정보다
#    ② 라벨이 필요 없는 값(f1·macro4·nsv·train_prior)은 **그냥 회수하면 된다**
#  그래서 (a) 07-17 실행 **자신의** 데이터 캐시에서 라벨을 찾고, (b) 못 찾으면 저장된
#  지표만 회수하고, (c) 어느 쪽이든 차이의 원인을 **레코드 단위로 진단**한다.
#  추측으로 정렬을 맞추지 않는다 — 안 맞으면 안 맞는다고 적고 잠근 채로 둔다.
import re
from sklearn.metrics import roc_auc_score

def to_rec(g):
    """그룹 배열 → 레코드 번호(int).

    ★ 실측: data_mit.npz 의 gp·gv·gt 는 dtype '<U9' 인 **문자열**이었다
      ('mitdb_100' 꼴). 정수만 받는 코드는 이걸 '레코드 ID 없음' 으로 오판한다.
      고유값에만 정규식을 돌리고 매핑한다(수만 개를 하나씩 훑지 않는다).
    """
    a = np.asarray(g)
    if np.issubdtype(a.dtype, np.integer):
        return a.astype(int)
    s = a.astype(str)
    m = {}
    for x in np.unique(s):
        dg = re.findall(r"\d+", x)
        m[x] = int(dg[-1]) if dg else -1
    return np.array([m[x] for x in s], dtype=int)

# ── 0) Drive 를 **한 번만** 훑어 색인. (이전 판은 파일마다 os.walk 를 다시 돌았다)
IDX = {}
for _root, _, _files in os.walk(DRIVE_ROOT):
    for _f in _files:
        IDX.setdefault(_f, []).append(os.path.join(_root, _f))

def one(name):
    """이름이 **유일할 때만** 경로를 준다. 중복이면 어느 쪽을 읽는지 불확실 → 안 쓴다."""
    p = IDX.get(name, [])
    if len(p) == 1:
        return p[0]
    if len(p) > 1:
        run.log(f"  ⚠️ '{name}' 이 {len(p)}곳에 있다 → 모호해서 사용하지 않는다: {p[:3]}")
    return None

def flat(key, v):
    """저장된 지표를 {이름: 스칼라} 로 편다. 모양을 **추측하지 않고** 있는 그대로."""
    if isinstance(v, dict):
        return {f"{key}.{k}": float(x) for k, x in v.items() if np.ndim(x) == 0}
    if np.ndim(v) == 0:
        return {key: float(v)}
    a = np.ravel(np.asarray(v, dtype="float64"))
    return {f"{key}[{j}]": float(a[j]) for j in range(a.size)}

paths = {n: one(n) for n in PKL_NAMES}
found = {n: p for n, p in paths.items() if p}
run.log(f"\n【G1】 확률 pkl {len(found)}/{len(PKL_NAMES)}개 발견")
G1 = None
META = None           # ★ 아래 두 블록이 공유한다. 미정의로 새면 NameError 로 조용히 죽는다
G1_NOTE = []          # ★ 카드·ASSETS_DRIVE 에 그대로 옮길 진단 문장

if len(found) < len(PKL_NAMES):
    run.log(f"  ⚠️ 없는 것: {[n for n in PKL_NAMES if n not in found]}")
    run.log("  → G1 **건너뛴다**. 인용 대기 표는 잠긴 상태로 둔다"
            " (주가설 P-A~P-D 는 G1 없이 성립한다)")
    G1_NOTE.append("pkl 일부 없음 → G1 미실행")
else:
    run.log(f"  경로 예: {list(found.values())[0]}")

    # ── 1) 라벨이 **필요 없는** 것 먼저 회수한다(재계산이 아니라 '저장된 값 회수')
    META = {}
    for n, p in found.items():
        o = pickle.load(open(p, "rb"))
        if not (isinstance(o, dict) and "proba" in o):
            run.log(f"  ⚠️ {n}: dict/proba 구조가 아니다 → G1 중단(추측 금지)")
            META = None
            break
        _pa = np.asarray(o["proba"])
        m = {"n_test": int(_pa.shape[0]),
             "n_cls": int(_pa.shape[1]) if _pa.ndim == 2 else -1}
        for k in ("arm", "seed", "train_prior", "nsv", "n_train", "n_S_train",
                  "n_pat_train", "mins"):
            if k in o:
                m[k] = o[k]
        for k in ("f1", "macro4"):
            if k in o:
                m.update(flat(k, o[k]))
        META[n] = m

if len(found) == len(PKL_NAMES) and META:
    run.log("\n  ── 저장된 메타(라벨 불필요) ─────────────────────────────")
    for n in PKL_NAMES:
        m = META[n]
        run.log(f"  {n:<22} n_test={m['n_test']:>7,} · n_train={m.get('n_train','?'):>7} "
                f"· n_S_train={m.get('n_S_train','?'):>6} · n_pat={m.get('n_pat_train','?')} "
                f"· {m.get('mins','?')}분")
    run.log(f"  train_prior 예시: {META[PKL_NAMES[0]].get('train_prior')}")

    # ── 1b) 【R10-b】 **길이는 정렬의 증거가 아니다.** 클래스 구성을 먼저 본다.
    #   pkl 안에 이미 답이 있었다: 'macro4' 라는 키 이름은 07-17 이 **4클래스**였다는
    #   뜻이고, 'nsv' 는 N/S/V **셋만** 센 값이다. nsv 합 < n_test 면 차액이 곧
    #   제4클래스(F) 개수다. 이걸 안 보면 "부족 + 초과" 가 **상쇄돼** 겉보기 길이만
    #   가깝게 보인다. 최악의 경우 길이가 정확히 일치하면서 정렬은 완전히 어긋난다.
    _raw_nsv = META[PKL_NAMES[0]].get("nsv")
    NSV = np.ravel(np.asarray(_raw_nsv)).astype("int64") if _raw_nsv is not None else None
    if NSV is not None and NSV.size < 3:
        # 1차 실행 실측: nsv 는 **스칼라 0** 이었다. 클래스 개수 배열이 아니다.
        run.log(f"  ⚠️ nsv = {NSV.tolist()} — 원소가 {NSV.size}개다. **클래스 개수 배열이"
                " 아니다**(플래그거나 미기록). 클래스 대조에 쓰지 않는다")
        G1_NOTE.append(f"nsv 는 클래스 개수가 아님(값 {NSV.tolist()})")
        NSV = None

    NCLS = sorted({m["n_cls"] for m in META.values()})
    TPR = np.ravel(np.asarray(META[PKL_NAMES[0]].get("train_prior", [])))
    run.log(f"\n  proba 열 수 {NCLS}  ← 4 면 07-17 은 N/S/V/**F** 4클래스다")
    if len(NCLS) == 1 and NCLS[0] == 4:
        run.log("  ★ **4클래스 확정.** 이 실험은 3클래스(N/S/V)다 — 07-17 테스트셋에는"
                " 여기 없는 **F(융합박동)** 가 섞여 있다.")
        run.log("     그래서 겉보기 차이는 '부족(N/S/V) − 초과(F)' 의 **상쇄값**이다.")
        G1_NOTE.append("07-17 은 4클래스(proba 4열) · 이 실험은 3클래스")

    # ── 1c) 학습쪽 메타로 **분할이 같은지** 본다. 이게 nsv 를 대신하는 지문이다.
    _ntr = META[PKL_NAMES[0]].get("n_train")
    _npat = META[PKL_NAMES[0]].get("n_pat_train")
    _nS = META[PKL_NAMES[0]].get("n_S_train")
    if TPR.size and _ntr and _nS is not None:
        _chk = abs(float(TPR[1]) * float(_ntr) - float(_nS))
        run.log(f"  train_prior {np.round(TPR, 6).tolist()}")
        run.log(f"  정합성: train_prior[1] × n_train = {float(TPR[1])*float(_ntr):.1f}"
                f" vs n_S_train {_nS}  → 오차 {_chk:.2f}"
                f" {'✅ 메타 신뢰 가능' if _chk < 1 else '⚠️ 안 맞는다'}")
    _tr1 = np.isin(mpid, _DS1)
    _DS1_N, _DS1_PAT = int(_tr1.sum()), int(len(np.unique(mpid[_tr1])))
    if _npat is not None:
        run.log(f"\n  ★ 07-17 학습 = **{_npat}환자 · {_ntr:,}비트 · S {_nS}**")
        run.log(f"     이 실험 DS1 = {_DS1_PAT}환자 · {_DS1_N:,}비트 ·"
                f" S {int((my[_tr1] == 1).sum()):,}")
        if int(_npat) != _DS1_PAT:
            run.log(f"     → 학습에 DS1 {_DS1_PAT}개 중 **{_npat}개만** 썼다."
                    f" 나머지 {_DS1_PAT - int(_npat)}개는 **검증 분할**일 수 있다"
                    " — 아래 캐시 분할표로 확인한다.")
            run.log("        (이것만으로 'DS1→DS2 가 아니다' 라고 결론내지 않는다."
                    " 표준 프로토콜 + DS1 내부 검증분할이면 학습 환자 수는 22 미만이다.)")
            run.log("        어느 쪽이든 **학습 표본이 DS1 전체가 아니므로**, 이 실험의")
            run.log("        within 기준선과 같은 행에 놓을 수는 없다.")
            G1_NOTE.append(f"07-17 학습 {_npat}환자 < DS1 {_DS1_PAT}환자"
                           " (검증분할 가능성 — 분할표로 확인)")

    # 짝지은 비교의 **전제**: 두 arm 이 같은 테스트셋을 봐야 한다
    NT = sorted({m["n_test"] for m in META.values()})
    if len(NT) != 1:
        run.log(f"  ❌ arm/seed 마다 테스트셋 크기가 다르다 {NT} — 짝지은 비교 자체가 불가능")
        G1_NOTE.append(f"테스트셋 크기 불일치 {NT} → 비교 불가")
    else:
        n_test = NT[0]
        gap = int(TE.sum()) - n_test
        MY_NSV = [int((my[TE] == k).sum()) for k in range(3)]
        run.log(f"\n  07-17 테스트셋 {n_test:,}비트  vs  이 실험 DS2 {int(TE.sum()):,}비트"
                f"  (차이 {gap:+,})")
        run.log(f"  이 실험 DS2 N/S/V = {MY_NSV} (합 {sum(MY_NSV):,})  ← **실측**. "
                "외부에서 전달된 표와 다르면 이쪽이 맞다")
        G1_NOTE.append(f"07-17 테스트셋 {n_test:,} ≠ 이 실험 DS2 {int(TE.sum()):,} ({gap:+,})")
        G1_NOTE.append(f"이 실험 DS2 N/S/V={MY_NSV}")
        # '레코드 통째 누락' 가설의 **가장 값싼 반증 기준값**: 어떤 레코드를 빼면 S 가
        #   몇 개나 딸려 나가는가. 문헌 인용 없이 **우리 배열에서 직접 센다**(R10-c).
        _yT, _pT = my[TE], mpid[TE]
        _sr = sorted((int(((_yT == 1) & (_pT == r)).sum()), int(r)) for r in np.unique(_pT))
        _smax, _rmax = _sr[-1]
        run.log(f"  DS2 레코드별 S 개수: 최대 #{_rmax} {_smax:,}개"
                f" (전체 S 의 {_smax/max(MY_NSV[1],1):.1%}) · 최소 {_sr[0][0]}개")

        def diag_deficit(counts, src):
            """【R10-c】 클래스별 부족을 내고 '레코드 소실 vs 분산 손실' 을 가른다."""
            d = [MY_NSV[k] - int(counts[k]) for k in range(3)]
            run.log(f"  클래스별 차이(이 실험 − 07-17) N/S/V = {d}"
                    f"  → 부족 소계 {sum(d):+,}   [출처 {src}]")
            run.log("  ※ 겉보기 차이는 **부족(N/S/V) + 초과(F) 가 상쇄된 값**이다."
                    " 두 방향을 따로 봐야 한다.")
            if abs(d[1]) < _smax * 0.25:
                run.log(f"  → S 부족 {abs(d[1]):,}개는 최대 레코드({_smax:,})보다 훨씬 작다."
                        " 특정 레코드가 통째로 빠진 것으로는 설명이 안 된다"
                        " → **전 레코드 분산 손실** 쪽이다.")
                G1_NOTE.append("클래스 프로파일상 레코드 통째 누락 아님(분산 손실)")
            else:
                run.log(f"  → S 부족 {abs(d[1]):,}개가 최대 레코드({_smax:,})에 육박한다."
                        " **레코드 통째 소실** 을 의심해야 한다 — 아래 레코드별 표로 확인.")
                G1_NOTE.append("S 부족이 커서 레코드 소실 의심")
            G1_NOTE.append(f"클래스별 부족 N/S/V={d} [{src}]")
            return d

        d3 = diag_deficit(NSV, "pkl nsv") if (NSV is not None and NSV.size >= 3) else None
        if d3 is None:
            run.log("  → pkl 만으로는 클래스 구성을 못 센다(nsv 가 개수 배열이 아님)."
                    " 아래 data_mit.npz 히스토그램이 **유일한** 클래스 증거다.")

        # ── 2) 차이의 원인을 **레코드 단위로** 진단한다(한 레코드가 통째로 빠졌나?)
        yv = None
        dpath = one("data_mit.npz")
        if dpath is None:
            run.log("  ⚠️ data_mit.npz 를 못 찾았다 → 07-17 실행의 라벨을 복원할 수 없다")
            G1_NOTE.append("data_mit.npz 미발견 → AUROC 재계산 불가")
        else:
            dz = np.load(dpath, allow_pickle=False)
            run.log(f"  data_mit.npz {os.path.getsize(dpath)/1e6:.1f} MB · 키 {list(dz.files)}")
            # ── 캐시가 **이미 분할돼 있을 수 있다.** 실측: yp/gp · yv/gv · yt/gt
            #   (1차 실행에서 'y'·'pid' 라는 이름만 찾다가 놓쳤다. 이름을 가정하지 말고
            #    1차원 정수 라벨 배열을 전부 후보로 잡고 **길이로** 짝을 찾는다.)
            SPL = {}
            for k in dz.files:
                if not k.startswith("y"):
                    continue
                arr = dz[k]
                if getattr(arr, "ndim", 0) != 1:
                    continue
                sfx = k[1:]
                gk = next((c for c in (f"g{sfx}", f"pid{sfx}", "pid") if c in dz.files), None)
                SPL[sfx] = (k, gk, int(arr.shape[0]))
            if not SPL:
                run.log("  ⚠️ 'y*' 1차원 배열이 없다 → 라벨 복원 불가(추측하지 않는다)")
                G1_NOTE.append("data_mit.npz 에 라벨 배열 없음")
            else:
                run.log(f"  ★ 캐시가 **{len(SPL)}개로 분할**돼 있다:")
                for sfx, (yk, gk, n) in sorted(SPL.items(), key=lambda x: -x[1][2]):
                    yy = np.asarray(dz[yk]).astype(int)
                    h = np.bincount(yy, minlength=5)[:5]
                    ng = int(len(np.unique(to_rec(dz[gk])))) if gk else -1
                    run.log(f"    {yk:<4} g={str(gk):<4} n={n:>7,} · 클래스0~4={h.tolist()}"
                            f" · 그룹 {ng}")
                run.log(f"    합계 {sum(v[2] for v in SPL.values()):,}비트"
                        f" (이 실험 MIT-BIH 전체 {len(my):,})")

                # 테스트 분할 = 길이가 pkl 의 n_test 와 같은 것
                cands = [s for s, v in SPL.items() if v[2] == n_test]
                # 학습쪽 지문 = (비트·그룹·S) 가 pkl 메타와 같은 분할이 있는가
                want = (int(_ntr or -1), int(_npat or -1), int(_nS if _nS is not None else -1))
                trh = []
                for s, (yk, gk, n) in SPL.items():
                    yy = np.asarray(dz[yk]).astype(int)
                    fp = (n, int(len(np.unique(to_rec(dz[gk])))) if gk else -1, int((yy == 1).sum()))
                    if fp == want:
                        trh.append(s)
                    run.log(f"    지문 {yk}: {fp}" + ("  ← pkl 학습 메타와 일치 ✅" if fp == want else ""))
                run.log(f"    pkl 학습 메타 {want}")

                if not cands:
                    run.log(f"  ⚠️ 길이 {n_test:,} 인 분할이 없다 → 테스트 라벨을 못 고른다")
                    G1_NOTE.append(f"길이 {n_test:,} 분할 없음 → AUROC 생략")
                    zp = zy = cand = None
                    zh = np.zeros(4, int)
                    ok_len = ok_cls = False
                    _fp = "테스트 분할 미확정"
                elif len(cands) > 1:
                    run.log(f"  ⚠️ 길이가 같은 분할이 {cands} 로 여러 개 — 어느 것인지 모호하다")
                    G1_NOTE.append(f"테스트 분할 모호 {cands}")
                    zp = zy = cand = None
                    zh = np.zeros(4, int)
                    ok_len = ok_cls = False
                    _fp = "테스트 분할 모호"
                else:
                    tk, tg, _ = SPL[cands[0]]
                    zy = np.asarray(dz[tk]).astype(int)
                    zp = to_rec(dz[tg]) if tg else None   # ★ 문자열이어도 레코드 번호로
                    cand = zy
                    zh = np.bincount(zy, minlength=4)[:4]
                    run.log(f"  → 테스트 분할 = '{tk}' (그룹 '{tg}') · 길이 {len(zy):,}")
                    run.log(f"  후보 라벨 히스토그램 {zh.tolist()} (클래스 0..3)")
                    if d3 is None:             # pkl 이 못 준 클래스 구성을 캐시가 준다
                        d3 = diag_deficit(zh, f"data_mit.npz['{tk}']")
                    # 【R10-b】 길이 하나로는 인정하지 않는다. 학습쪽 지문이 두 번째 키다.
                    ok_len = True
                    ok_cls = bool(trh)
                    _fp = (f"학습 지문 일치 분할 {trh}" if trh
                           else f"학습 지문 일치 분할 없음 (pkl {want})")
                if ok_len and ok_cls:
                    yv = cand
                    run.log(f"  ✅ 길이 **와** 두 번째 증거가 모두 일치 ({_fp})"
                            " — 07-17 실행 **자신의** 캐시에서 라벨을 복원했다")
                    if int(zh[3]) > 0:
                        run.log(f"     (제4클래스 F {int(zh[3]):,}비트는 그대로 둔다."
                                " S·V 의 one-vs-rest AUROC 에서 '나머지' 로 들어간다 —")
                        run.log("      07-17 실행 자신이 한 것과 같은 조건이다)")
                        G1_NOTE.append(f"테스트셋 F {int(zh[3]):,}비트 포함")
                elif ok_len and not ok_cls:
                    run.log(f"  ❌ 길이는 맞는데 **두 번째 증거가 어긋난다** — {_fp}")
                    run.log("     이게 R10-b 가 잡으려는 경우다. 길이 일치는 정렬의 증거가 아니다.")
                    G1_NOTE.append(f"길이 일치·구성 불일치 → 정렬 거부 ({_fp})")
                else:
                    run.log("  ⚠️ 테스트 분할을 확정하지 못했다 → AUROC 생략(추측 금지)")

                # ── 레코드별 비교. 그룹 ID 가 **레코드 번호인지 먼저 확인**한다.
                #   0..N 인덱스일 수도 있는데, 그러면 전 레코드가 0 으로 보여
                #   '전부 누락' 이라는 거짓 진단이 나온다.
                if zp is None:
                    run.log("  (그룹 배열이 없어 레코드별 비교를 건너뛴다)")
                else:
                    gu = set(int(x) for x in np.unique(zp))   # zp 는 이미 to_rec 를 거쳤다
                    known = set(int(r) for r in list(_DS1) + list(_DS2))
                    ov = gu & known
                    run.log(f"  그룹 ID {len(gu)}종 · 예시 {sorted(gu)[:8]}"
                            f" · MIT-BIH 레코드번호와 겹침 {len(ov)}개")
                    # ★ 가장 결정적인 한 줄: 07-17 테스트가 **DS2 인가**.
                    #   DS1 레코드가 섞여 있으면 애초에 DS2 가 아니고, 그러면
                    #   '비트를 흘렸다' 가 아니라 **다른 분할**이 답이다.
                    _in1 = sorted(gu & set(int(r) for r in _DS1))
                    _in2 = sorted(gu & set(int(r) for r in _DS2))
                    run.log(f"    그중 DS1 레코드 {len(_in1)}개 {_in1[:8]}"
                            f" · DS2 레코드 {len(_in2)}개")
                    if _in1:
                        run.log("    ★ **DS1 레코드가 테스트에 있다 → 07-17 테스트셋은"
                                " DS2 가 아니다.** 비트 손실이 아니라 분할이 다른 것이다.")
                        G1_NOTE.append(f"07-17 테스트에 DS1 레코드 {len(_in1)}개 포함 → DS2 아님")
                    if not ov:
                        run.log("  ⚠️ 그룹 ID 가 **레코드 번호가 아니다**(인덱스로 보인다)."
                                " 레코드별 비교는 불가능 — 매핑을 지어내지 않는다.")
                        G1_NOTE.append(f"그룹 ID 가 레코드번호 아님(예시 {sorted(gu)[:5]})")
                    else:
                        tgt = sorted(ov)
                        a = {int(r): int((zp == r).sum()) for r in tgt}
                        b = {int(r): int((mpid == r).sum()) for r in tgt}
                        dif = {r: b[r] - a[r] for r in tgt if a[r] != b[r]}
                        miss = [int(r) for r in _DS2 if int(r) not in gu]
                        gone = [r for r in dif if a[r] == 0]
                        if miss:
                            run.log(f"  07-17 테스트에 **아예 없는 DS2 레코드** {miss}")
                            G1_NOTE.append(f"07-17 테스트에 없는 DS2 레코드 {miss}")
                        if not dif:
                            run.log("  겹치는 레코드의 비트 수는 **동일**")
                            G1_NOTE.append("겹치는 레코드 비트 수 동일")
                        else:
                            part = [r for r in dif if a[r] > 0]
                            run.log(f"  레코드별 차이(이 실험 − 07-17), 0 아닌 것만: {dif}")
                            run.log(f"    통째로 빠진 레코드 {gone if gone else '없음'}"
                                    f" · 부분 손실 {len(part)}개"
                                    f" (합 {sum(dif[r] for r in part):+,})")
                            G1_NOTE.append(f"누락 레코드 {gone} · 부분손실 {len(part)}개")

        # ── 3) 저장된 지표로 짝지은 비교 (라벨 불필요 · 항상 가능)
        common = set.intersection(*[{k for k, v in m.items()
                                     if isinstance(v, float)} for m in META.values()])
        common = sorted(k for k in common if k.startswith(("f1", "macro4")))
        if common:
            run.log("\n  ── 저장된 지표의 짝지은 Δ (mit_svdb − mit_only, 시드 5쌍) ──")
            run.log(f"  {'지표':<14}{'mit_only':>11}{'mit_svdb':>11}{'Δ':>11}{'95% CI':>22}")
            for k in common:
                v0 = np.array([META[f"mit_only_{1000+i}.pkl"][k] for i in range(5)])
                v1 = np.array([META[f"mit_svdb_{1000+i}.pkl"][k] for i in range(5)])
                m_, lo, hi = t_ci(v1 - v0)
                run.log(f"  {k:<14}{v0.mean():>11.4f}{v1.mean():>11.4f}{m_:>+11.4f}"
                        f"   [{lo:+.4f}, {hi:+.4f}]")
            run.log("  ⚠️ f1·macro4 는 **동작점(임계값)에 붙은** 지표다(R2·R4 계열). Δ 안에")
            run.log("     판별력 변화와 임계값 이동이 섞여 있다 → AUROC 를 대체하지 못한다.")
            run.log("     ASSETS_DRIVE.md 의 '인용 대기' 표(PR-AUC Δ)는 이걸로 해금되지 않는다.")

        # ── 4) 라벨이 복원됐을 때만 AUROC 재계산
        if yv is not None:
            G1 = {}
            for arm in ("mit_only", "mit_svdb"):
                A = {"S": [], "V": []}
                for i in range(5):
                    p = np.asarray(pickle.load(open(found[f"{arm}_{1000+i}.pkl"], "rb"))["proba"],
                                   dtype="float64")
                    A["S"].append(roc_auc_score((yv == 1).astype(int), p[:, 1]))
                    A["V"].append(roc_auc_score((yv == 2).astype(int), p[:, 2]))
                G1[arm] = A
            run.log(f"\n  ── AUROC 재계산 (07-17 자체 테스트셋 {n_test:,}비트) ──")
            run.log(f"  {'클래스':<6}{'mit_only':>12}{'mit_svdb':>12}{'Δ':>12}{'95% CI':>24}")
            for c in ("S", "V"):
                d = np.array(G1["mit_svdb"][c]) - np.array(G1["mit_only"][c])
                m_, lo, hi = t_ci(d)
                run.log(f"  {c:<6}{np.mean(G1['mit_only'][c]):>12.4f}"
                        f"{np.mean(G1['mit_svdb'][c]):>12.4f}{m_:>+12.4f}"
                        f"   [{lo:+.4f}, {hi:+.4f}]")
            run.log("  ※ 이건 **데이터 증강**(DS1+SVDB → DS2) 이지 교차DB 낙폭이 아니다.")
            run.log("     '축별 상반 반응' 으로 명명한다 — V 는 오르고 S 는 내리는가?")
            run.log(f"  ※ 테스트셋이 이 실험의 DS2({int(TE.sum()):,})와 다르므로 P-D 표에")
            run.log("     **같은 행으로 넣지 않는다**. 별도 코호트로 각주 처리한다.")

CONFIG["g1"] = {"note": G1_NOTE,
                "auroc_recovered": G1 is not None,
                "meta": {n: {k: (v if isinstance(v, (int, float, str)) else str(v))
                             for k, v in META[n].items()} for n in PKL_NAMES} if META else None}
run.save_json("config", CONFIG)


In [ ]:
# CELL 3b — 【G1-b】 S 는 **판별력이 나빠졌나, 임계값이 옮겨졌나**
#
#  CELL 3 이 낸 것:  F1 −0.1229 (0 제외) · AUROC −0.0203 (0 포함).
#  두 지표가 갈렸다 → 이걸 가르지 않으면 결론을 못 쓴다.
#
#    · F1        : **임계값 의존**. 동작점이 옮겨가면 판별력이 그대로여도 떨어진다
#    · AUROC     : 순위 기반, 전체 구간
#    · PR-AUC    : 순위 기반, **상위 구간에 가중**
#    · R-precision(상위 k, k = 실제 양성 수) : 순위 기반, **딱 상위 k만**
#
#  판독은 **유의성이 아니라 효과크기 비**로 한다:
#      REL = max|Δ 순위지표| / |Δ F1|
#    REL 이 작다 → F1 변화를 순위가 **설명하지 못한다** → 임계값 이동
#    REL 이 크다 → 순위가 같이 움직였다 → 판별력 변화
#
#  ★ 왜 유의성으로 안 하나: 픽스처에서 **순수 임계값 이동**을 넣었더니 PR-AUC Δ 가
#    −0.0003 인데 CI 가 [−0.0004, −0.0003] 로 0 을 제외해 '유의' 가 떴다. 시드 5개가
#    거의 결정론적이라 CI 가 비현실적으로 좁다. **유의하지만 무의미한 값**이다(R5 계열).
#    REL 로 보면 0.0003/0.3765 = 0.08% 로 즉시 갈린다.
#
#  왜 이게 중요한가: mit_svdb 는 학습 S 사전확률이 mit_only 의 **10.2배**다
#  (0.01102 → 0.11226). 테스트 실제 S 비율은 0.03782 이므로 mit_only 는 3.4배
#  적게, mit_svdb 는 3.0배 많게 배웠다. 임계값이 안 움직였을 리가 없다.
#
#  ★ 덤으로 **무결성 검증**: 저장된 f1[1]·f1[2] 를 확률+복원라벨로 재계산해 맞춰본다.
#    맞으면 라벨 복원이 끝에서 끝까지 옳았다는 증거다.
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

if yv is None or G1 is None:
    run.log("\n【G1-b】 라벨이 복원되지 않아 건너뛴다")
else:
    CLS = {"S": 1, "V": 2}
    NPOS = {c: int((yv == i).sum()) for c, i in CLS.items()}
    run.log("\n" + "=" * 108)
    run.log("【G1-b】 순위 지표 vs 임계값 지표 — S 의 하락은 어느 쪽인가")
    run.log("=" * 108)
    run.log(f"  테스트 {len(yv):,}비트 · 실제 양성 " +
            " · ".join(f"{c} {NPOS[c]:,}({NPOS[c]/len(yv):.2%})" for c in CLS))
    _po = META["mit_only_1000.pkl"]; _ps = META["mit_svdb_1000.pkl"]
    _ro = _po["n_S_train"] / _po["n_train"]; _rs = _ps["n_S_train"] / _ps["n_train"]
    run.log(f"  학습 S 사전확률 mit_only {_ro:.5f} · mit_svdb {_rs:.5f}"
            f" → **{_rs/_ro:.1f}배**. 테스트 실제 {NPOS['S']/len(yv):.5f}")
    run.log("  → 동작점이 안 움직였을 리가 없다. 그래서 임계값 지표로는 판정하지 않는다.")

    M = {}                       # M[arm][지표][클래스] = 시드 5개 리스트
    F1CHK = {}
    for arm in ("mit_only", "mit_svdb"):
        acc = {k: {c: [] for c in CLS} for k in ("auroc", "prauc", "rprec")}
        f1a = {c: [] for c in CLS}
        for i in range(5):
            o = pickle.load(open(found[f"{arm}_{1000+i}.pkl"], "rb"))
            p = np.asarray(o["proba"], dtype="float64")
            pred = p.argmax(1)                      # 저장된 f1 과 같은 동작점
            for c, idx in CLS.items():
                t = (yv == idx).astype(int)
                s = p[:, idx]
                acc["auroc"][c].append(roc_auc_score(t, s))
                acc["prauc"][c].append(average_precision_score(t, s))
                k = NPOS[c]                          # 상위 k = 실제 양성 수
                top = np.argpartition(-s, k - 1)[:k]
                acc["rprec"][c].append(float(t[top].sum()) / k)
                f1a[c].append(f1_score(t, (pred == idx).astype(int), zero_division=0))
        M[arm] = acc
        F1CHK[arm] = f1a

    # ── 무결성 검증: 재계산 F1 vs 저장된 f1[]
    run.log("\n  ── 무결성 검증 (재계산 F1 vs pkl 에 저장된 f1) ──")
    okall = True
    for arm in ("mit_only", "mit_svdb"):
        for c, idx in CLS.items():
            mine = float(np.mean(F1CHK[arm][c]))
            keyd = [META[f"{arm}_{1000+i}.pkl"].get(f"f1[{idx}]") for i in range(5)]
            if any(v is None for v in keyd):
                run.log(f"    {arm} {c}: 저장된 f1[{idx}] 없음 — 대조 생략"); continue
            theirs = float(np.mean(keyd)); dd = abs(mine - theirs)
            okall &= dd < 5e-4
            run.log(f"    {arm:<9} {c}  재계산 {mine:.4f}  저장 {theirs:.4f}"
                    f"  차이 {dd:.2e} {'✅' if dd < 5e-4 else '❌'}")
    run.log(f"  → 라벨 복원 {'**끝에서 끝까지 검증됨**' if okall else '⚠️ 불일치 — 정렬 재검토'}")
    G1_NOTE.append(f"F1 재계산 대조 {'일치' if okall else '불일치'}")

    # ── 본표
    NAMES = [("auroc", "AUROC", "순위·전체"), ("prauc", "PR-AUC", "순위·상위가중"),
             ("rprec", f"R-prec", "순위·상위k만")]
    run.log(f"\n  {'클래스':<5}{'지표':<10}{'성격':<14}{'mit_only':>10}{'mit_svdb':>10}"
            f"{'Δ':>10}{'95% CI':>22}  판정")
    VERDICT = {}
    for c in CLS:
        rows = []
        for key, lab, kind in NAMES:
            v0 = np.array(M["mit_only"][key][c]); v1 = np.array(M["mit_svdb"][key][c])
            m_, lo, hi = t_ci(v1 - v0)
            sig = lo * hi > 0
            rows.append((lab, m_, sig))
            run.log(f"  {c:<5}{lab:<10}{kind:<14}{v0.mean():>10.4f}{v1.mean():>10.4f}"
                    f"{m_:>+10.4f}   [{lo:+.4f}, {hi:+.4f}]  "
                    f"{'유의' if sig else '미결'}")
        # 임계값 지표(저장된 F1)도 같은 표에
        idx = CLS[c]
        f0 = np.array([META[f"mit_only_{1000+i}.pkl"][f"f1[{idx}]"] for i in range(5)])
        f1_ = np.array([META[f"mit_svdb_{1000+i}.pkl"][f"f1[{idx}]"] for i in range(5)])
        mf, lof, hif = t_ci(f1_ - f0)
        fsig = lof * hif > 0
        run.log(f"  {c:<5}{'F1@argmax':<10}{'**임계값**':<14}{f0.mean():>10.4f}"
                f"{f1_.mean():>10.4f}{mf:>+10.4f}   [{lof:+.4f}, {hif:+.4f}]  "
                f"{'유의' if fsig else '미결'}")
        rmax = max(abs(m) for _, m, _ in rows)
        rlab = max(rows, key=lambda x: abs(x[1]))[0]
        REL = rmax / max(abs(mf), 1e-9)
        run.log(f"  {'':<5}{'효과크기비':<10}{'':<14}"
                f"max|Δ순위| {rmax:.4f} ({rlab}) / |ΔF1| {abs(mf):.4f}"
                f"  → REL = **{REL:.1%}**")
        if not fsig and rmax < 0.01:
            v = "변화 없음"
            run.log(f"  → {c}: F1 도 순위도 안 움직였다.")
        elif not fsig:
            v = "순위만 변화"
            run.log(f"  → {c}: F1 은 미결인데 순위는 움직였다 — 드문 조합. 값을 직접 읽을 것.")
        elif REL < 0.20 and rmax < 0.02:
            v = "임계값 이동"
            run.log(f"  → {c}: **F1 변화를 순위가 설명하지 못한다**(REL {REL:.1%}, "
                    f"max|Δ순위| {rmax:.4f}).")
            run.log(f"       판별력이 아니라 **동작점** 문제다 → 임계값 재보정으로 복구 가능.")
        elif REL > 0.50:
            v = "판별력 변화"
            run.log(f"  → {c}: **순위가 F1 만큼(또는 그 이상) 움직였다**(REL {REL:.1%})"
                    " → 진짜 판별력 변화다. 재보정으로 복구 안 된다.")
        else:
            v = "혼합"
            run.log(f"  → {c}: REL {REL:.1%} — **혼합**. 임계값 이동과 판별력 변화가"
                    " 함께 있다. 재보정으로 일부만 복구된다.")
        VERDICT[c] = v
        G1_NOTE.append(f"{c} 판정: {v}")
        run.log("")

    run.log("  ※ PR-AUC·R-prec 는 **순위 기반**이라 확률을 단조변환해도 안 변한다.")
    run.log("     사전확률 shift 는 단조변환이므로 이 둘을 못 움직인다 — 그래서 가른다.")
    run.log("  ※ 판정은 **유의성이 아니라 효과크기 비(REL)** 로 한다. 시드가 거의")
    run.log("     결정론적이면 무의미한 차이에도 CI 가 0 을 제외한다(픽스처에서 실제로 발생).")
    CONFIG["g1b"] = {"verdict": VERDICT, "f1_recheck_ok": bool(okall),
                     "prior_ratio": float(_rs / _ro)}
    run.save_json("config", CONFIG)


In [ ]:
# CELL 3c — 【G1-c】 S 손상이 **한 환자짜리인가**, 그리고 무엇이 상위를 채웠나
#
#  G1-b 결론: S 는 PR-AUC −0.2194 · R-prec −0.2094 로 **상위 순위가 오염**됐다.
#  그런데 테스트 S 1,808개 중 **1,381개(76%)가 #232 한 명**이다(PAPER.md 관찰3).
#  → 이 발견이 "코호트 성질" 인지 "환자 한 명" 인지 가르지 않으면 쓸 수 없다.
#
#  두 가지를 한다:
#    ① #232 를 빼고 다시 잰다. 남는 S 는 427개(21레코드). 여기서도 무너지면 코호트 성질
#    ② 상위 k 목록에 **무엇이 들어왔는지** 센다 — 진짜 S 가 379개 빠진 자리를
#       어떤 클래스·어떤 레코드가 채웠나. 이게 '음의 전이' 의 실체다
#
#  ⚠️ R4: #232 제외 코호트와 전체 코호트의 **PR-AUC 값을 서로 비교하지 않는다**
#     (기저율이 다르다). 각 코호트 **안에서의 짝지은 Δ** 만 읽는다.
from sklearn.metrics import roc_auc_score, average_precision_score

if yv is None or G1 is None or zp is None:
    run.log("\n【G1-c】 라벨 또는 레코드 ID 가 없어 건너뛴다")
else:
    SIDX, VIDX = 1, 2
    rec = np.asarray(zp)
    run.log("\n" + "=" * 108)
    run.log("【G1-c】 S 손상이 한 환자짜리인가 · 상위 목록을 무엇이 채웠나")
    run.log("=" * 108)

    _sc = sorted(((int(((yv == SIDX) & (rec == r)).sum()), int(r))
                  for r in np.unique(rec)), reverse=True)
    DOM, DOMN = _sc[0][1], _sc[0][0]
    nS = int((yv == SIDX).sum())
    run.log(f"  테스트 S {nS:,}개 · 최다 보유 **#{DOM} {DOMN:,}개 ({DOMN/nS:.1%})**"
            f" · 2위 #{_sc[1][1]} {_sc[1][0]:,}개")

    def metrics(mask, idx):
        """마스크 안에서 arm×seed 별 (AUROC, PR-AUC, R-prec)."""
        t = (yv[mask] == idx).astype(int)
        k = int(t.sum())
        out = {}
        for arm in ("mit_only", "mit_svdb"):
            a = {"auroc": [], "prauc": [], "rprec": []}
            for i in range(5):
                p = np.asarray(pickle.load(open(found[f"{arm}_{1000+i}.pkl"], "rb"))["proba"],
                               dtype="float64")[mask, idx]
                a["auroc"].append(roc_auc_score(t, p) if 0 < k < len(t) else np.nan)
                a["prauc"].append(average_precision_score(t, p) if k else np.nan)
                top = np.argpartition(-p, k - 1)[:k] if 0 < k < len(p) else np.arange(len(p))
                a["rprec"].append(float(t[top].sum()) / max(k, 1))
            out[arm] = a
        return out, k

    ALL = np.ones(len(yv), bool)
    COH = [("전체", ALL), (f"#{DOM} 제외", rec != DOM), (f"#{DOM} 만", rec == DOM)]
    run.log(f"\n  {'코호트':<12}{'양성 n':>7}{'지표':<10}{'mit_only':>10}{'mit_svdb':>10}"
            f"{'Δ':>10}{'95% CI':>22}")
    G1C = {}
    for nm, m in COH:
        if m.sum() == 0:
            continue
        M3, k = metrics(m, SIDX)
        if k == 0 or k == int(m.sum()):
            run.log(f"  {nm:<12}{k:>7,}  → 한 클래스뿐이라 채점 불가. 건너뜀")
            continue
        row = {}
        for key, lab in (("auroc", "AUROC"), ("prauc", "PR-AUC"), ("rprec", "R-prec")):
            v0 = np.array(M3["mit_only"][key]); v1 = np.array(M3["mit_svdb"][key])
            mm, lo, hi = t_ci(v1 - v0)
            row[key] = (mm, lo, hi)
            run.log(f"  {nm:<12}{k:>7,}  {lab:<10}{np.nanmean(v0):>10.4f}"
                    f"{np.nanmean(v1):>10.4f}{mm:>+10.4f}   [{lo:+.4f}, {hi:+.4f}]"
                    f"  {'유의' if lo*hi > 0 else '미결'}")
        G1C[nm] = {k2: [float(x) for x in v] for k2, v in row.items()}
        run.log("")

    ex = G1C.get(f"#{DOM} 제외", {}).get("prauc")
    fu = G1C.get("전체", {}).get("prauc")
    if ex and fu:
        run.log(f"  ★ 판정: #{DOM} 를 빼도 PR-AUC Δ = {ex[0]:+.4f} [{ex[1]:+.4f}, {ex[2]:+.4f}]")
        if ex[1] * ex[2] > 0 and ex[0] < 0:
            run.log(f"     → **한 환자짜리가 아니다.** {DOM} 없이도 상위 순위가 무너진다"
                    " = 코호트 성질.")
            G1_NOTE.append(f"S 손상은 #{DOM} 제외해도 유지 → 한 환자짜리 아님")
        elif abs(ex[0]) < 0.02:
            run.log(f"     → ❗ #{DOM} 를 빼면 **효과가 사라진다**(|Δ| {abs(ex[0]):.4f})."
                    f" 이 발견은 **#{DOM} 한 명의 성질**이다. 일반화 금지.")
            G1_NOTE.append(f"S 손상은 #{DOM} 한 명에서만 — 일반화 불가")
        elif ex[0] < 0:
            run.log(f"     → 방향은 같지만 **미결**. 남은 양성 {len(yv[rec != DOM]):,}중"
                    " 소수라 검정력이 부족하다 — 한 환자짜리를 배제하지 못한다.")
            G1_NOTE.append(f"S 손상 #{DOM} 제외 시 미결(검정력 부족)")
        else:
            run.log(f"     → ❗ #{DOM} 를 빼면 **부호가 뒤집힌다**(Δ {ex[0]:+.4f})."
                    f" 이 발견은 **#{DOM} 한 명의 성질**이다. 일반화 금지.")
            G1_NOTE.append(f"S 손상은 #{DOM} 한 명에서만(부호 반전) — 일반화 불가")

    # ── ② 상위 k 목록을 무엇이 채웠나 (시드 평균)
    k = nS
    run.log(f"\n  ── S 점수 상위 {k:,}개의 **정체** (시드 5개 평균) ──")
    comp = {}
    for arm in ("mit_only", "mit_svdb"):
        cls = np.zeros(4); byrec = {}
        for i in range(5):
            p = np.asarray(pickle.load(open(found[f"{arm}_{1000+i}.pkl"], "rb"))["proba"],
                           dtype="float64")[:, SIDX]
            top = np.argpartition(-p, k - 1)[:k]
            cls += np.bincount(yv[top], minlength=4)[:4] / 5.0
            for r, cnt in zip(*np.unique(rec[top], return_counts=True)):
                byrec[int(r)] = byrec.get(int(r), 0.0) + cnt / 5.0
        comp[arm] = (cls, byrec)
        run.log(f"    {arm:<9} 진짜 N {cls[0]:>7.1f} · **S {cls[1]:>7.1f}** ·"
                f" V {cls[2]:>7.1f} · F {cls[3]:>6.1f}")
    d = comp["mit_svdb"][0] - comp["mit_only"][0]
    run.log(f"    {'변화':<9} N {d[0]:>+7.1f} · **S {d[1]:>+7.1f}** · V {d[2]:>+7.1f}"
            f" · F {d[3]:>+6.1f}")
    run.log(f"    → 상위 {k:,} 자리에서 진짜 S 가 **{-d[1]:.0f}개 빠지고**"
            f" 그 자리를 N {d[0]:+.0f} · V {d[2]:+.0f} · F {d[3]:+.0f} 가 채웠다")
    G1_NOTE.append(f"상위{k} 에서 진짜 S {-d[1]:.0f}개 감소 · N{d[0]:+.0f} V{d[2]:+.0f} F{d[3]:+.0f}")

    gain = sorted(((comp["mit_svdb"][1].get(r, 0) - comp["mit_only"][1].get(r, 0), r)
                   for r in set(comp["mit_only"][1]) | set(comp["mit_svdb"][1])),
                  reverse=True)
    run.log(f"    상위 목록이 **늘어난** 레코드: "
            + " · ".join(f"#{r} {g:+.0f}" for g, r in gain[:5]))
    run.log(f"    상위 목록이 **줄어든** 레코드: "
            + " · ".join(f"#{r} {g:+.0f}" for g, r in gain[-5:]))
    run.log("\n  ※ R4: 코호트가 달라지면 PR-AUC 값 자체는 비교하지 않는다."
            " 각 코호트 **안의 짝지은 Δ** 만 읽었다.")
    CONFIG["g1c"] = {"dominant_record": DOM, "dominant_share": DOMN / nS, "by_cohort": G1C}
    run.save_json("config", CONFIG)


In [ ]:
# CELL 3d — 【G1-d】 무너진 건 **환자 안의 판별력**인가 **환자 간 점수 눈금**인가
#
#  G1-c 가 보여준 것:
#    · #232 **안에서는** mit_svdb 가 오히려 낫다 (PR-AUC +0.0053 · R-prec +0.0130)
#    · 그런데 **전역** PR-AUC 는 −0.2194 로 무너진다
#    · 상위 1,808 목록에서 #210(+272) · #219(+182) · #221(+43) 가 슬롯을 가져갔는데
#      이 셋의 진짜 S 는 9 · 7 · 0 개다 → **최소 588개가 위양성**
#
#  → 가설: 환자 **안의** 순위는 멀쩡한데, 환자 **사이의 점수 눈금**이 깨졌다.
#    mit_svdb 가 특정 레코드에 전반적으로 높은 S 점수를 주고, 그게 다른 레코드의
#    진짜 S 를 전역 목록에서 밀어낸다.
#
#  검정 방법: 점수를 **레코드 안에서 순위정규화**한 뒤 전역 지표를 다시 잰다.
#    · 레코드 내 순위정규화는 환자별 단조 변환을 전부 제거한다
#      → 환자 **안의** 순위는 100% 보존, 환자 **간** 눈금 차이만 사라진다
#    · 정규화 후 Δ 가 0 으로 돌아오면 **눈금 문제**(고칠 수 있다)
#    · 그대로 남으면 **판별력 문제**(고칠 수 없다)
#
#  ★ 이건 배포에서 실제로 가능한 처치다. 홀터 분석은 어느 기록을 보고 있는지 알기
#    때문에 레코드 내 정규화는 라벨 없이 쓸 수 있다(우리 파이프라인의 `_medref`·
#    환자상대 리듬특징과 같은 발상이다).
from sklearn.metrics import average_precision_score

GMIN_REC = 20      # 레코드 매크로 평균에 넣을 최소 S 개수 (R7)

if yv is None or G1 is None or zp is None:
    run.log("\n【G1-d】 라벨 또는 레코드 ID 가 없어 건너뛴다")
else:
    SIDX = 1
    rec = np.asarray(zp)
    t = (yv == SIDX).astype(int)
    nS = int(t.sum())
    run.log("\n" + "=" * 108)
    run.log("【G1-d】 환자 안 판별력 vs 환자 간 점수 눈금")
    run.log("=" * 108)

    recs = np.unique(rec)
    idx_by_rec = {int(r): np.where(rec == r)[0] for r in recs}
    keep = [r for r in idx_by_rec if int(t[idx_by_rec[r]].sum()) >= GMIN_REC]
    run.log(f"  레코드 {len(recs)}개 · S ≥ {GMIN_REC} 인 레코드 {len(keep)}개 {sorted(keep)}"
            f" (여기 S {sum(int(t[idx_by_rec[r]].sum()) for r in keep):,}"
            f"/{nS:,} = {sum(int(t[idx_by_rec[r]].sum()) for r in keep)/nS:.1%})")

    def rank_in_rec(s):
        """레코드 **안에서** 순위를 [0,1] 로. 환자 간 눈금만 제거하고 환자 안 순위는 보존."""
        out = np.empty(len(s))
        for r, ii in idx_by_rec.items():
            v = s[ii]
            out[ii] = (v.argsort().argsort() + 0.5) / len(v) if len(v) > 1 else 0.5
        return out

    def rprec(s, tt, k):
        top = np.argpartition(-s, k - 1)[:k]
        return float(tt[top].sum()) / k

    ACC = {a: {k: [] for k in ("pr_raw", "rp_raw", "pr_norm", "rp_norm", "macro_rp")}
           for a in ("mit_only", "mit_svdb")}
    PERREC = {}
    for arm in ("mit_only", "mit_svdb"):
        for i in range(5):
            s = np.asarray(pickle.load(open(found[f"{arm}_{1000+i}.pkl"], "rb"))["proba"],
                           dtype="float64")[:, SIDX]
            ACC[arm]["pr_raw"].append(average_precision_score(t, s))
            ACC[arm]["rp_raw"].append(rprec(s, t, nS))
            sn = rank_in_rec(s)
            ACC[arm]["pr_norm"].append(average_precision_score(t, sn))
            ACC[arm]["rp_norm"].append(rprec(sn, t, nS))
            # 레코드 매크로: 각 레코드 안에서만 채점하고 평균 (환자 간 비교가 아예 없다)
            per = [rprec(s[idx_by_rec[r]], t[idx_by_rec[r]], int(t[idx_by_rec[r]].sum()))
                   for r in keep]
            ACC[arm]["macro_rp"].append(float(np.mean(per)))
            PERREC.setdefault(arm, []).append(per)      # [seed][rec] — 부트스트랩용

    run.log(f"\n  {'처치':<26}{'지표':<10}{'mit_only':>10}{'mit_svdb':>10}"
            f"{'Δ':>10}{'95% CI':>22}")
    OUT = {}
    for key, proc, lab in (("pr_raw", "원본(전역)", "PR-AUC"),
                           ("rp_raw", "원본(전역)", "R-prec"),
                           ("pr_norm", "레코드내 순위정규화", "PR-AUC"),
                           ("rp_norm", "레코드내 순위정규화", "R-prec"),
                           ("macro_rp", f"레코드 매크로(S≥{GMIN_REC})", "R-prec")):
        v0 = np.array(ACC["mit_only"][key]); v1 = np.array(ACC["mit_svdb"][key])
        m_, lo, hi = t_ci(v1 - v0)
        OUT[key] = (float(m_), float(lo), float(hi))
        run.log(f"  {proc:<26}{lab:<10}{v0.mean():>10.4f}{v1.mean():>10.4f}"
                f"{m_:>+10.4f}   [{lo:+.4f}, {hi:+.4f}]  "
                f"{'유의' if lo*hi > 0 else '미결'}")

    raw, nrm = OUT["pr_raw"][0], OUT["pr_norm"][0]
    mac, mlo, mhi = OUT["macro_rp"]
    rec_frac = 1 - abs(nrm) / max(abs(raw), 1e-9)

    # ★ 주 판정은 **레코드 매크로**가 한다. 이 지표는 각 레코드 안에서만 채점하므로
    #   환자 간 비교가 **정의상 0** 이다 → 환자 안 판별력을 단독으로 잰다.
    #   회복비율(rec_frac)은 '전역 하락 중 눈금이 설명하는 몫' 을 재는 **보조** 값이다.
    #   (픽스처에서 배운 것: rec_frac 만 보면 환자 안 손상을 '혼합' 으로 흐린다.)
    run.log(f"\n  ★ 레코드 매크로 R-prec Δ = **{mac:+.4f}** [{mlo:+.4f}, {mhi:+.4f}]"
            "  ← 환자 간 비교가 **아예 없는** 채점 = 주 판정")
    mac_bad = (mac < -0.02) and (mlo * mhi > 0)
    if not mac_bad:
        v = "환자 간 눈금"
        run.log("     → 환자 **안의** 순위는 멀쩡하다(또는 나아졌다).")
        run.log("        전역 하락은 **환자 간 점수 눈금**이 깨져서 생긴 것이다.")
        run.log("        배포에서 레코드 내 정규화(라벨 불필요)로 복구할 수 있다.")
    elif rec_frac > 0.5:
        v = "환자 안 판별력(+눈금)"
        run.log("     → **환자 안 판별력이 손상됐다.** 다만 전역 하락에는 눈금 몫도 크다.")
    else:
        v = "환자 안 판별력"
        run.log("     → **환자 안 판별력이 손상됐다.** 정규화로도 안 돌아온다.")
    run.log(f"     보조: 레코드내 정규화 회복비율 1 − |{nrm:+.4f}|/|{raw:+.4f}|"
            f" = {rec_frac:.1%}  (전역 하락 중 **눈금**이 설명하는 몫)")
    G1_NOTE.append(f"G1-d 판정: {v} (정규화 회복 {rec_frac:.0%})")

    # ── 【R8】 매크로는 레코드 7개 평균이다. 시드 CI 는 **레코드 표집 변동을 못 잡는다**.
    #   레코드를 복원추출해 부트스트랩 CI 를 따로 낸다. 넓은 쪽으로 판정한다.
    D = (np.asarray(PERREC["mit_svdb"], float) -
         np.asarray(PERREC["mit_only"], float)).mean(0)      # 레코드별 Δ(시드평균)
    run.log(f"\n  ── 레코드별 R-prec Δ (시드 5개 평균) · 레코드 {len(keep)}개 ──")
    for r, d_ in sorted(zip(keep, D), key=lambda x: -x[1]):
        run.log(f"    #{r}  S {int(t[idx_by_rec[r]].sum()):>5,}개  Δ {d_:+.4f}")
    B = 5000
    rs = np.random.RandomState(0)
    bs = np.array([D[rs.randint(0, len(D), len(D))].mean() for _ in range(B)])
    blo, bhi = np.percentile(bs, [2.5, 97.5])
    run.log(f"\n  레코드 부트스트랩({B:,}회) 매크로 Δ = {D.mean():+.4f}"
            f"  [{blo:+.4f}, {bhi:+.4f}]")
    run.log(f"  시드 CI 폭 {mhi-mlo:.4f} vs 레코드 부트스트랩 폭 {bhi-blo:.4f}"
            f"  → **{'레코드' if bhi-blo > mhi-mlo else '시드'}** 쪽이 넓다 (R8: 넓은 쪽으로 판정)")
    wide_ok = blo * bhi > 0
    npos = int(np.sum(D > 0))
    run.log(f"  레코드 {len(D)}개 중 개선 {npos}개 · 악화 {len(D)-npos}개"
            f"  → 부호검정 최소 p = {2**(-len(D)):.4f}")
    if not wide_ok:
        run.log(f"  ⚠️ **넓은 쪽 CI 가 0 을 포함한다.** 레코드가 적어({len(D)}개) 매크로 개선을")
        run.log("     단정할 수 없다. '전역 하락이 눈금 탓' 까지는 서지만,")
        run.log("     '증강이 S 를 개선했다' 는 **더 많은 환자에서 확인해야 한다**.")
    else:
        run.log("  ✅ 레코드 부트스트랩에서도 0 을 제외한다 — 레코드 표집에 강건하다.")
    G1_NOTE.append(f"매크로 레코드부트스트랩 [{blo:+.4f}, {bhi:+.4f}] · 개선 {npos}/{len(D)}")
    CONFIG.setdefault("g1d_boot", {}).update(
        {"delta": float(D.mean()), "lo": float(blo), "hi": float(bhi),
         "n_rec": int(len(D)), "n_improved": npos,
         "per_record": {int(r): float(d_) for r, d_ in zip(keep, D)}})

    run.log("\n  ※ 레코드 내 순위정규화는 환자 안 순위를 100% 보존한다. 따라서 이 처치로")
    run.log("     회복되는 부분은 **정의상 환자 간 눈금**이고, 안 되는 부분이 판별력이다.")
    CONFIG["g1d"] = {"verdict": v, "recovered_frac": float(rec_frac),
                     "gmin_rec": GMIN_REC, "n_rec_scored": len(keep), **OUT}
    run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — DS1 학습 → DS2(within) + INCART(cross) **같은 모델로** 예측
#   ★ colab_crossdb.py 의 함수를 그대로 쓰되 **학습 분할만** 바꾼다.
#     백본·특징·하이퍼파라미터는 손대지 않는다(그래야 §6.5 와 이어붙는다).
import torch
# ★ 두 파일을 **순서대로** 읽는다. split 은 crossdb 의 module-level 심볼에 의존한다.
for _f in ("colab_crossdb.py", "colab_crossdb_split.py"):
    _p = os.path.join(MITBIH, _f)
    if not os.path.exists(_p):
        raise AssetError(f"{_f} 가 없다: {_p}\n"
                         "  → colab_crossdb_split.py 는 실험22-A 에서 추가된 파일이다. "
                         "repo mit-bih/ 에서 Drive 로 올려야 한다")
    exec(open(_p).read(), globals())
globals()["_BASE"] = MITBIH
globals()["_FEATDIR"] = f"{MITBIH}/synergy_feats"

# split 이 쓰는 심볼이 전부 들어왔는지 확인 — 없으면 학습 전에 멈춘다
_need = ["_DS1", "_DS2", "_determinism", "_znorm", "_medref", "_net",
         "_incart_feats", "_crossdb_rhythm", "auto_weights", "run_crossdb_split"]
_lack = [n for n in _need if n not in globals()]
if _lack:
    raise AssetError(f"exec 후에도 {_lack} 이 없다.\n"
                     "  → Drive 의 colab_crossdb.py 가 repo 사본과 다를 수 있다. "
                     "추측하지 않고 멈춘다.")
run.log("하니스 적재 ✅ colab_crossdb.py + colab_crossdb_split.py")

# ★ 캐시 이름에 시드 수를 박는다. 안 그러면 스모크(1시드) 결과가 본 실행(5시드)을
#   가로채고, CELL 5 는 그걸 모른 채 채점한다.
CACHE = run.data(f"exp22a_probs_s{len(SEEDS)}_v1.npz")
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    P = {k: z[k] for k in z.files}
    run.log(f"\n예측 캐시 적중 {CACHE} · {list(P)}")
else:
    run.log("\n학습 시작 — DS1 만으로 학습해 DS2·INCART 를 같은 모델로 예측한다")
    t0 = time.time()
    P = run_crossdb_split(seeds=SEEDS, Kwst=KWST, train_mask=TR, test_mask=TE)
    np.savez_compressed(CACHE, **P)
    run.log(f"완료 {time.time()-t0:.0f}s → 캐시 저장")

# P 는 {"<구성>_<within|cross>_<raw|bn>": (시드, 비트, 3)} 형태여야 한다
run.log("\n예측 배열")
for k in sorted(P):
    run.log(f"  {k:<28}{np.shape(P[k])}")
EXPECT = [f"{c}_{s}_raw" for c in ("v1", "v2") for s in ("within", "cross")] + \
         ["y_within", "y_cross"]
lack = [k for k in EXPECT if k not in P]
if lack:
    raise AssetError(f"예측 배열에 {lack} 이 없다 · 있는 것 {sorted(P)}\n"
                     "  → run_crossdb_split 이 기대한 형식을 안 냈다. 채점으로 넘어가지 않는다.")


In [ ]:
# CELL 5 — 【P-A·P-B·P-C】 AUROC 낙폭 (R4 — 코호트를 가로지르므로 AUROC 만)
CLS = {"S": 1, "V": 2}
# ★ 라벨은 예측과 **같은 출처**에서 받는다(정렬이 어긋나면 조용히 다른 실험이 된다)
Y = {"within": np.asarray(P["y_within"]), "cross": np.asarray(P["y_cross"])}
for sp in ("within", "cross"):
    n_p, n_y = np.shape(P[f"v1_{sp}_raw"])[1], len(Y[sp])
    if n_p != n_y:
        raise AssetError(f"{sp}: 예측 {n_p}행 vs 라벨 {n_y}행 — 정렬이 안 맞는다")
run.log(f"\n라벨 정렬 확인 ✅ within {len(Y['within']):,} · cross {len(Y['cross']):,}")

def auc_(y, s, k):
    yy = (y == k).astype(int)
    return float(roc_auc_score(yy, s)) if yy.any() and (yy == 0).any() else np.nan

AUC, DROP = {}, {}
for cfg in ("v1", "v2"):
    for mode in ("raw", "bn"):
        key = f"{cfg}_{{}}_{mode}"
        if key.format("within") not in P or key.format("cross") not in P:
            continue
        for c, k in CLS.items():
            w = [auc_(Y["within"], P[key.format("within")][i][:, k], k)
                 for i in range(len(SEEDS))]
            x = [auc_(Y["cross"], P[key.format("cross")][i][:, k], k)
                 for i in range(len(SEEDS))]
            AUC[(cfg, mode, c, "within")] = w
            AUC[(cfg, mode, c, "cross")] = x
            DROP[(cfg, mode, c)] = [w[i] - x[i] for i in range(len(w))]

run.log("\n" + "=" * 108)
run.log("【P-A·P-B·P-C】 AUROC — within(MIT-BIH DS2) vs cross(INCART) · 같은 학습")
run.log("=" * 108)
run.log(f"  {'구성':<6}{'적응':<6}{'클래스':<6}{'within':>10}{'cross':>10}{'낙폭':>10}{'95% CI':>22}")
for (cfg, mode, c), d in sorted(DROP.items()):
    mw, _, _ = t_ci_logit(AUC[(cfg, mode, c, "within")])
    mx, _, _ = t_ci_logit(AUC[(cfg, mode, c, "cross")])
    m, lo, hi = t_ci(d)
    run.log(f"  {cfg:<6}{mode:<6}{c:<6}{mw:>10.4f}{mx:>10.4f}{m:>+10.4f}"
            f"   [{lo:+.4f}, {hi:+.4f}]")

# P-A: V 낙폭 < S 낙폭 (리듬 없는 v1 에서 — 형태 축만 준 상태가 가장 깨끗한 대비)
gapA = [DROP[("v1", "raw", "S")][i] - DROP[("v1", "raw", "V")][i] for i in range(len(SEEDS))]
mA, lA, hA = t_ci(gapA)
run.log(f"\n  P-A  S 낙폭 − V 낙폭 (v1·raw) = {mA:+.4f} [{lA:+.4f}, {hA:+.4f}] "
        f"vs 문턱 {GAP_THR}")
run.log("       V 는 넓은 QRS 라는 **형태**로 정의되고 S 는 '평소보다 이르다' 는")
run.log("       **개인 기준 상대량**이다. 같은 실행 안에서 V 가 양성 대조군 역할을 한다")

# P-B: 리듬 축이 S 의 낙폭을 줄이나
gapB = None
if ("v2", "raw", "S") in DROP:
    gapB = [DROP[("v1", "raw", "S")][i] - DROP[("v2", "raw", "S")][i] for i in range(len(SEEDS))]
    mB, lB, hB = t_ci(gapB)
    run.log(f"\n  P-B  S 낙폭(v1) − S 낙폭(v2) = {mB:+.4f} [{lB:+.4f}, {hB:+.4f}]")
    run.log("       양수면 **리듬 축이 전이를 구해준다**(§6.5 의 1.8x → 7.9x 를 AUROC 로 재현)")

# P-C: BN 적응이 복구하나
gapC = None
if ("v2", "bn", "S") in DROP:
    gapC = [DROP[("v2", "raw", "S")][i] - DROP[("v2", "bn", "S")][i] for i in range(len(SEEDS))]
    mC, lC, hC = t_ci(gapC)
    run.log(f"\n  P-C  S 낙폭(raw) − S 낙폭(BN 적응) = {mC:+.4f} [{lC:+.4f}, {hC:+.4f}]")
    run.log("       0 근처면 **입력 분포 shift 가 아니라 판별 축의 문제**다(§6.5 결론 재현)")


In [ ]:
# CELL 5′ — 【P-A′·P-B′】 **R11 로 본 실험을 다시 채점한다**
#
#  왜: P-A~P-D 를 **전역 AUROC** 로 냈는데, G1-c/G1-d 가 전역 지표의 함정을 보여줬다.
#      MIT-BIH DS2 는 #232 한 명이 S 의 76% 를 갖는다 → 전역 S 지표는 사실상 한 명이다.
#      INCART 도 지배 지분을 재보기 전에는 모른다.
#  → R11: **주 지표는 환자 매크로**. 여기서 낙폭을 다시 내고 전역과 비교한다.
#
#  G1-d 결론(레코드내 정규화로 전역 하락의 95.4% 가 회복)에 따라 **정규화 행도** 낸다.
#  단 주의: 정규화는 '이 기록은 S 가 많다' 는 **진짜 정보도 버린다**(G1-d 에서 mit_only
#  PR-AUC 0.5138 → 0.0851). 그래서 정규화는 **처방이 아니라 진단 도구**로 쓴다.
#
#  세 가지를 낸다:
#    ① 코호트별 **지배 지분** — 전역 지표를 단독 인용해도 되는지 판정(R11-3)
#    ② **환자 매크로 AUROC** 낙폭 vs 전역 낙폭
#    ③ 레코드 부트스트랩 CI (R8 — 시드 CI 는 환자 표집을 못 잡는다)
from sklearn.metrics import roc_auc_score

GMIN_S = 20          # 환자별 채점 최소 양성 수 (R7 · R11-2)
NBOOT = 5000

CLS = {"S": 1, "V": 2}
COH = {"within": (np.asarray(P["y_within"]), np.asarray(mpid[TE])),
       "cross":  (np.asarray(P["y_cross"]),  np.asarray(ipid))}
run.log("\n" + "=" * 108)
run.log("【P-A′·P-B′】 R11 재채점 — 전역이 아니라 **환자 매크로**로")
run.log("=" * 108)

# ── ① 지배 지분 (R11-3): 전역 지표를 단독으로 인용해도 되는가
run.log(f"  {'코호트':<8}{'클래스':<5}{'양성':>8}{'환자':>6}{'최대환자':>10}{'지배지분':>10}"
        f"{'채점환자':>10}  전역 단독인용")
DOMSH = {}
KEEP = {}
for cn, (y, g) in COH.items():
    for c, idx in CLS.items():
        pos = (y == idx)
        per = {int(r): int((pos & (g == r)).sum()) for r in np.unique(g)}
        tot = int(pos.sum())
        mx = max(per.values()) if per else 0
        share = mx / max(tot, 1)
        keep = [r for r, n in per.items() if n >= GMIN_S and n < int((g == r).sum())]
        DOMSH[(cn, c)] = share
        KEEP[(cn, c)] = keep
        run.log(f"  {cn:<8}{c:<5}{tot:>8,}{len(per):>6}"
                f"{max(per, key=per.get):>10}{share:>9.1%}{len(keep):>10}"
                f"  {'❌ 금지' if share > 0.5 else '✅ 가능'}")
run.log(f"  (채점 환자 = 양성 ≥ {GMIN_S} 이고 음성도 있는 환자)")

# ── ② 환자 매크로 AUROC
def macro_auroc(prob, y, g, keep, idx):
    """환자별 AUROC 를 내고 (매크로평균, 환자별배열) 반환. 환자 간 비교가 없다."""
    out = []
    for r in keep:
        m = g == r
        t = (y[m] == idx)                    # ★ bool 로 둔다. int 에 ~ 를 쓰면 비트연산이다
        # ★ `(~t).all() is False` 로 쓰면 np.bool_ 와 파이썬 False 의 **동일성** 비교라
        #   항상 거짓이 되어 전부 NaN 이 된다(픽스처에서 실제로 걸렸다).
        ok = bool(t.any()) and not bool(t.all())
        out.append(roc_auc_score(t.astype(int), prob[m]) if ok else np.nan)
    a = np.asarray(out, float)
    if not np.isfinite(a).any():
        raise RuntimeError("채점 가능한 환자가 하나도 없다 — keep 조건을 확인할 것")
    return float(np.nanmean(a)), a

run.log(f"\n  {'구성':<5}{'적응':<5}{'클래스':<5}{'채점':<8}"
        f"{'within':>9}{'cross':>9}{'낙폭':>9}   {'시드 95% CI':>22}")
MAC = {}
for cfg in ("v1", "v2"):
    for ad in ("raw", "bn"):
        for c, idx in CLS.items():
            kw, kc = KEEP[("within", c)], KEEP[("cross", c)]
            if not kw or not kc:
                continue
            gw, gc = np.asarray(mpid[TE]), np.asarray(ipid)
            yw, yc = np.asarray(P["y_within"]), np.asarray(P["y_cross"])
            dw, dc, drops = [], [], []
            pw_all, pc_all = P[f"{cfg}_within_{ad}"], P[f"{cfg}_cross_{ad}"]
            perw, perc = [], []
            for s in range(pw_all.shape[0]):
                mw, aw = macro_auroc(pw_all[s][:, idx], yw, gw, kw, idx)
                mc, ac = macro_auroc(pc_all[s][:, idx], yc, gc, kc, idx)
                dw.append(mw); dc.append(mc); drops.append(mw - mc)
                perw.append(aw); perc.append(ac)
            m_, lo, hi = t_ci(np.asarray(drops))
            # 전역(원 P-A) 과 나란히
            gw_glob = float(np.mean([roc_auc_score((yw == idx).astype(int), pw_all[s][:, idx])
                                     for s in range(pw_all.shape[0])]))
            gc_glob = float(np.mean([roc_auc_score((yc == idx).astype(int), pc_all[s][:, idx])
                                     for s in range(pc_all.shape[0])]))
            MAC[(cfg, ad, c)] = {"macro": (np.mean(dw), np.mean(dc), m_, lo, hi),
                                 "drops": np.asarray(drops),
                                 "global": (gw_glob, gc_glob, gw_glob - gc_glob),
                                 "perw": np.mean(perw, 0), "perc": np.mean(perc, 0)}
            run.log(f"  {cfg:<5}{ad:<5}{c:<5}{f'{len(kw)}/{len(kc)}':<8}"
                    f"{np.mean(dw):>9.4f}{np.mean(dc):>9.4f}{m_:>+9.4f}"
                    f"   [{lo:+.4f}, {hi:+.4f}]  {'유의' if lo*hi>0 else '미결'}")

# ── 전역 vs 매크로 나란히 (결론이 바뀌는지)
run.log(f"\n  ── 전역 낙폭 vs 매크로 낙폭 (같은 확률, 채점만 다름) ──")
run.log(f"  {'구성':<5}{'적응':<5}{'클래스':<5}{'전역 낙폭':>11}{'매크로 낙폭':>13}{'차이':>10}")
for k, v in MAC.items():
    g_ = v["global"][2]; m_ = v["macro"][2]
    run.log(f"  {k[0]:<5}{k[1]:<5}{k[2]:<5}{g_:>+11.4f}{m_:>+13.4f}{m_-g_:>+10.4f}")

# ── ③ 사전등록 관문 재채점 + 레코드 부트스트랩 (R8)
run.log("\n  ── 【P-A′】 S 낙폭 − V 낙폭 (v1·raw) · 매크로 기준 ──")
if ("v1", "raw", "S") in MAC and ("v1", "raw", "V") in MAC:
    dS, dV = MAC[("v1", "raw", "S")]["macro"][2], MAC[("v1", "raw", "V")]["macro"][2]
    gS, gV = MAC[("v1", "raw", "S")]["global"][2], MAC[("v1", "raw", "V")]["global"][2]
    run.log(f"    전역   S {gS:+.4f} − V {gV:+.4f} = {gS-gV:+.4f}   (원래 보고값)")
    run.log(f"    매크로 S {dS:+.4f} − V {dV:+.4f} = {dS-dV:+.4f}")
    run.log(f"    → 부호 {'**유지**' if (gS-gV)*(dS-dV) > 0 else '❗**반전**'}"
            "  (문턱 0.05 · 방향은 S 낙폭 > V 낙폭 이면 지지)")

run.log(f"\n  ── 【R8】 환자 부트스트랩 CI ({NBOOT:,}회) — 시드 CI 는 환자 표집을 못 잡는다 ──")
_rs = np.random.RandomState(0)
for k, v in MAC.items():
    aw, ac = v["perw"], v["perc"]
    bw = np.array([np.nanmean(aw[_rs.randint(0, len(aw), len(aw))]) -
                   np.nanmean(ac[_rs.randint(0, len(ac), len(ac))]) for _ in range(NBOOT)])
    blo, bhi = np.nanpercentile(bw, [2.5, 97.5])
    slo, shi = v["macro"][3], v["macro"][4]
    wider = "환자" if (bhi - blo) > (shi - slo) else "시드"
    run.log(f"  {k[0]:<5}{k[1]:<5}{k[2]:<5} 매크로 낙폭 {v['macro'][2]:+.4f}"
            f"  시드[{slo:+.4f},{shi:+.4f}] 폭 {shi-slo:.4f}"
            f"  · 환자[{blo:+.4f},{bhi:+.4f}] 폭 {bhi-blo:.4f}"
            f"  → **{wider}** 넓음")

# ── 【P-B′·P-C′】 나머지 관문도 매크로로 재채점 (부호가 바뀌는지)
GATES = {}
def gate(name, a, b, thr, why):
    """a − b 를 시드 CI 와 **환자 부트스트랩** 양쪽으로 재고 넓은 쪽으로 판정(R8)."""
    da, db = MAC[a], MAC[b]
    d = da["drops"] - db["drops"]
    m_, lo, hi = t_ci(d)
    # 환자 부트스트랩: 같은 환자 인덱스를 두 구성에 **짝지어** 적용한다
    rs = np.random.RandomState(1)
    nw, nc = len(da["perw"]), len(da["perc"])
    bb = []
    for _ in range(NBOOT):
        iw, ic = rs.randint(0, nw, nw), rs.randint(0, nc, nc)
        f = lambda v: np.nanmean(v["perw"][iw]) - np.nanmean(v["perc"][ic])
        bb.append(f(da) - f(db))
    blo, bhi = np.nanpercentile(bb, [2.5, 97.5])
    wide = (blo, bhi) if (bhi - blo) > (hi - lo) else (lo, hi)
    v = decide(wide[0], wide[1], thr, ">")
    v = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}[v]
    run.log(f"\n  {name}  = {m_:+.4f}")
    run.log(f"    시드 [{lo:+.4f}, {hi:+.4f}]  ·  환자 [{blo:+.4f}, {bhi:+.4f}]"
            f"  → 넓은 쪽 [{wide[0]:+.4f}, {wide[1]:+.4f}] vs 문턱 {thr}")
    run.log(f"    {v}   {why}")
    return m_, v

if all(k in MAC for k in (("v1","raw","S"), ("v2","raw","S"), ("v2","bn","S"))):
    run.log("\n  ── 【P-B′·P-C′】 매크로 재채점 (R8: 넓은 쪽으로 판정) ──")
    pb, vb = gate("P-B′ 리듬이 S 낙폭을 줄이나  S낙폭(v1) − S낙폭(v2)",
                  ("v1","raw","S"), ("v2","raw","S"), 0.0,
                  "양수면 §6.5 의 '리듬이 전이를 구해준다' 가 AUROC 매크로에서 재현")
    pc, vc = gate("P-C′ BN 적응이 복구하나  S낙폭(raw) − S낙폭(bn)",
                  ("v2","raw","S"), ("v2","bn","S"), 0.0,
                  "0 근처면 입력분포 shift 가 아니라 판별축 문제 (§6.5 결론 재현)")
    GATES.update({"P_B_macro": float(pb), "P_B_verdict": vb,
                  "P_C_macro": float(pc), "P_C_verdict": vc})

run.log("\n  ※ R11-3: 지배 지분 > 50% 인 코호트·클래스는 **전역 지표를 단독 인용하지 않는다.**")
run.log("  ※ 이 표가 P-D 의 근거를 대체한다 — 카드에는 매크로를 주로, 전역을 보조로 적는다.")
CONFIG["r11_rescore"] = {
    "gmin_s": GMIN_S,
    "dominance": {f"{a}_{b}": float(s) for (a, b), s in DOMSH.items()},
    "n_scored": {f"{a}_{b}": len(v) for (a, b), v in KEEP.items()},
    "macro_vs_global": {f"{a}_{b}_{c}": {"global": float(v["global"][2]),
                                         "macro": float(v["macro"][2])}
                        for (a, b, c), v in MAC.items()},
    **GATES}          # ★ 마지막 대입이 관문 결과를 덮어쓰던 것 수정(dry-run 에서 발견)
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【P-D】 축 비교표 · 사전등록 채점
run.log("\n" + "=" * 108)
run.log("【P-D】 축별 낙폭 — 같은 자(AUROC)로")
run.log("=" * 108)
run.log(f"  {'축':<26}{'과제':<16}{'낙폭(AUROC)':>14}{'출처':>22}")
for s, d in MI_DROPS.items():
    run.log(f"  {'형태 (레코드·12유도)':<26}{'MI ' + s:<16}{d:>+14.4f}{'실험20':>22}")
for c, nm in (("V", "심실조기박동"), ("S", "심방조기박동")):
    m, lo, hi = t_ci(DROP[("v1", "raw", c)])
    run.log(f"  {'비트 (형태특징만)':<26}{nm:<16}{m:>+14.4f}{'실험22-A':>22}")
if ("v2", "raw", "S") in DROP:
    for c, nm in (("V", "심실조기박동"), ("S", "심방조기박동")):
        m, _, _ = t_ci(DROP[("v2", "raw", c)])
        run.log(f"  {'비트 (형태+리듬)':<26}{nm:<16}{m:>+14.4f}{'실험22-A':>22}")
run.log(f"\n  ⚠️ {MI_NOTE}")
run.log("  ⚠️ 비트 단위와 레코드 단위의 **성능**은 비교하지 않는다 — 낙폭만 비교한다")

run.log("\n" + "=" * 108)
run.log("【사전등록 채점】")
run.log("=" * 108)
V = {}
V["P-A"] = decide(lA, hA, GAP_THR, ">")
run.log(f"\n  P-A S 낙폭 − V 낙폭 > {GAP_THR}")
run.log(f"      {mA:+.4f} [{lA:+.4f}, {hA:+.4f}] → {MARK[V['P-A']]}")
run.log("      지지 = **전이 강건성은 축에 따라 다르다.** 타이밍은 기기 불변이 아니고")
run.log("             형태는 기기 불변이다 — 우리 직관과 반대일 수 있다")
if gapB is not None:
    V["P-B"] = decide(lB, hB, 0.0, ">")
    run.log(f"\n  P-B 리듬 축이 S 의 낙폭을 줄이나 (> 0)")
    run.log(f"      {mB:+.4f} [{lB:+.4f}, {hB:+.4f}] → {MARK[V['P-B']]}")
if gapC is not None:
    V["P-C"] = decide(lC, hC, 0.0, ">")
    run.log(f"\n  P-C BN 적응이 복구하나 (> 0)")
    run.log(f"      {mC:+.4f} [{lC:+.4f}, {hC:+.4f}] → {MARK[V['P-C']]}")
    run.log("      **미결/기각이면 §6.5 의 '판별 축 부재' 결론이 AUROC 에서도 재현**된다")
run.log("\n" + "  ".join(f"{k}: {MARK[v]}" for k, v in sorted(V.items())))
run.log(f"  G1: {'재계산 완료' if G1 else '건너뜀(자산 미발견 또는 구조 불일치)'}")
run.log("  ⚠️ §6.5 의 한계 L1·L2·L4·L5 는 그대로 남는다. L3 만 부분 교정했다")
run.log("  ⚠️ 시드 5개(t 배수 2.776)")


In [ ]:
# CELL 7 — 그림 + 저장
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))

labs, vals, cols = [], [], []
for c, nm in (("V", "비트 V\n(형태 정의)"), ("S", "비트 S\n(타이밍 정의)")):
    labs.append(nm); vals.append(np.mean(DROP[("v1", "raw", c)])); cols.append("#2ca02c" if c == "V" else "#d62728")
for s, d in MI_DROPS.items():
    labs.append(f"MI {s}\n(형태·레코드)"); vals.append(d); cols.append("#999999")
ax[0].bar(range(len(vals)), vals, color=cols)
ax[0].set_xticks(range(len(vals))); ax[0].set_xticklabels(labs, fontsize=8)
ax[0].axhline(0, c="k", lw=.8); ax[0].set_ylabel("AUROC 낙폭 (within − cross)")
ax[0].set_title(f"축별 전이 낙폭 · P-A {MARK[V['P-A']]}")

if gapB is not None:
    xs = np.arange(2); w = 0.35
    ax[1].bar(xs - w/2, [np.mean(DROP[("v1", "raw", c)]) for c in ("V", "S")], w,
              label="v1 형태만", color="#bbbbbb")
    ax[1].bar(xs + w/2, [np.mean(DROP[("v2", "raw", c)]) for c in ("V", "S")], w,
              label="v2 +리듬", color="#1f77b4")
    ax[1].set_xticks(xs); ax[1].set_xticklabels(["V", "S"])
    ax[1].axhline(0, c="k", lw=.8); ax[1].legend(fontsize=8)
    ax[1].set_ylabel("AUROC 낙폭"); ax[1].set_title(f"리듬 축의 구조 효과 · P-B {MARK[V.get('P-B')]}")
plt.tight_layout(); run.save_fig("exp22a_axis_transfer", fig); plt.show()

res = {
    "week": 2, "exp_id": "exp22a_axis", "quest": "ailab-2026-0015",
    "step": "exp22a-axis-transfer", "split": "inter",
    "task": "축별 전이 강건성을 같은 자(AUROC)로 모은다",
    "notebook": "notebooks/exp22a_axis_transfer.ipynb",
    "metric": "drop_gap_S_minus_V", "value": round(float(mA), 4),
    "passed": bool(V.get("P-A") is True),
    "seeds": SEEDS,
    "auroc": {f"{c}|{m}|{cl}|{sp}": float(np.mean(v)) for (c, m, cl, sp), v in AUC.items()},
    "drop": {f"{c}|{m}|{cl}": [float(x) for x in v] for (c, m, cl), v in DROP.items()},
    "mi_drops_reference": MI_DROPS, "mi_note": MI_NOTE,
    "g1_recomputed": (None if G1 is None else
                      {a: {c: [float(x) for x in G1[a][c]] for c in ("S", "V")} for a in G1}),
    "verdicts": {k: V[k] for k in V},
    "caveats": [
        "R4 — 코호트를 가로지르는 비교는 AUROC 로만. PR-AUC·lift 는 천장이 다르다",
        "비트 단위와 레코드 단위의 성능은 비교하지 않는다 — 낙폭만",
        "§6.5 의 한계 L1·L2·L4·L5 는 그대로. L3 만 부분 교정",
        "형태 축(MI) 낙폭은 내부 5겹 OOF vs 외부 전량 학습이라 과소추정",
        ("DS1 S 기저율 0.0187 vs DS2 0.0373 — within 기준선에도 이미 2배 유병률 이동이 "
         "있다. AUROC 는 무관하나 within 을 이상적 상한으로 읽으면 안 된다"),
        "G1 은 데이터 증강 실험이지 교차DB 낙폭이 아니다 — '축별 상반 반응'"],
}
run.save_json("result", res); run.finish(res)
print(json.dumps({k: res[k] for k in ("metric", "value", "verdicts")},
                 ensure_ascii=False, indent=2))
